In [1]:
from pathlib import Path
import timeit
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import accuracy_score, recall_score

PROJECT_ROOT = Path(r"C:\tinyml-fdia-windows")

KERAS_MODEL = PROJECT_ROOT / "models" / "keras" / "LSTM_HPO.keras"
DATASET_PATH = PROJECT_ROOT / "data" / "fdia_dataset_processed.npz"

TFLITE_MODELS = {
    "Baseline": Path("LSTM_Original_frozen.tflite"),
    "Dynamic": Path("LSTM_Original_dynamic.tflite"),
}

# Load Keras model
keras_model = tf.keras.models.load_model(
    KERAS_MODEL,
    compile=False
)

# Load dataset
data = np.load(DATASET_PATH)

X_test = data["X_test"]
y_test = data["y_test"]

X_test_lstm = X_test.transpose(
    0, 2, 1
).astype(np.float32)

# Fixed 10% timing subset
rng = np.random.default_rng(42)

timing_indices = rng.choice(
    len(X_test_lstm),
    size=int(len(X_test_lstm) * 0.10),
    replace=False
)

X_timing = X_test_lstm[timing_indices]

results = []

for model_name, model_path in TFLITE_MODELS.items():

    interpreter = tf.lite.Interpreter(
        model_path=str(model_path)
    )

    interpreter.allocate_tensors()

    input_detail = interpreter.get_input_details()[0]
    output_detail = interpreter.get_output_details()[0]

    # Keras vs TFLite numerical check
    max_difference = 0.0

    for i in range(20):
        sample = X_test_lstm[i:i+1]

        keras_output = keras_model(
            sample,
            training=False
        ).numpy()

        interpreter.set_tensor(
            input_detail["index"],
            sample
        )

        interpreter.invoke()

        tflite_output = interpreter.get_tensor(
            output_detail["index"]
        )

        difference = np.max(
            np.abs(keras_output - tflite_output)
        )

        max_difference = max(
            max_difference,
            difference
        )

    # Full test-set metrics
    y_pred = []

    for sample in X_test_lstm:
        sample = sample[np.newaxis, ...]

        interpreter.set_tensor(
            input_detail["index"],
            sample
        )

        interpreter.invoke()

        output = interpreter.get_tensor(
            output_detail["index"]
        )

        y_pred.append(
            int(output[0][0] >= 0.5)
        )

    y_pred = np.array(y_pred)

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    fdia_recall = recall_score(
        y_test,
        y_pred,
        pos_label=1
    )

    fault_recall = recall_score(
        y_test,
        y_pred,
        pos_label=0
    )

    # Warm-up
    for i in range(10):
        sample = X_timing[i:i+1]

        interpreter.set_tensor(
            input_detail["index"],
            sample
        )

        interpreter.invoke()

        interpreter.get_tensor(
            output_detail["index"]
        )

    # Timing
    times = []

    for sample in X_timing:
        sample = sample[np.newaxis, ...]

        start = timeit.default_timer()

        interpreter.set_tensor(
            input_detail["index"],
            sample
        )

        interpreter.invoke()

        output = interpreter.get_tensor(
            output_detail["index"]
        )

        end = timeit.default_timer()

        times.append(
            (end - start) * 1000
        )

    results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "FDIA Recall": fdia_recall,
        "Fault Recall": fault_recall,
        "Max Keras Difference": max_difference,
        "Size (KB)": model_path.stat().st_size / 1024,
        "Inference Mean (ms)": np.mean(times),
        "Inference Std (ms)": np.std(times),
    })

results_df = pd.DataFrame(results)

print(results_df.to_string(index=False))

c:\tinyml-fdia-windows\venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
c:\tinyml-fdia-windows\venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


   Model  Accuracy  FDIA Recall  Fault Recall  Max Keras Difference  Size (KB)  Inference Mean (ms)  Inference Std (ms)
Baseline  0.989198     0.987847      0.990415              0.000001 681.566406             4.199149            0.561334
 Dynamic  0.989095     0.987847      0.990219              0.005192 246.031250             3.745156            0.556040


In [1]:
from pathlib import Path
import timeit
import numpy as np
import keras

PROJECT_ROOT = Path(r"C:\tinyml-fdia-windows")

MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "keras"
    / "LSTM_HPO.keras"
)

DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "fdia_dataset_processed.npz"
)

model = keras.models.load_model(
    MODEL_PATH,
    compile=False
)

data = np.load(DATASET_PATH)

X_test = data["X_test"]
y_test = data["y_test"]

X_test_lstm = X_test.transpose(
    0, 2, 1
).astype(np.float32)

print("Keras version:", keras.__version__)
print("Model loaded.")
print("Input:", X_test_lstm.shape)

Keras version: 3.15.1
Model loaded.
Input: (9720, 83, 6)


In [2]:
QUANT_MODEL_PATH = Path(
    "LSTM_Original_Keras_INT8.keras"
)

model.quantize("int8")

model.save(
    QUANT_MODEL_PATH
)

print("INT8 quantization completed.")
print(
    f"Quantized model size: "
    f"{QUANT_MODEL_PATH.stat().st_size / 1024:.3f} KB"
)

c:\tinyml-fdia-windows\venv\Lib\site-packages\keras\src\models\model.py:547: UserWarning: Layer InputLayer does not have a `quantize` method implemented.
  warnings.warn(str(e))
c:\tinyml-fdia-windows\venv\Lib\site-packages\keras\src\models\model.py:547: UserWarning: Layer LSTMCell does not have a `quantize` method implemented.
  warnings.warn(str(e))


INT8 quantization completed.
Quantized model size: 623.920 KB


In [3]:
QUANT_MODEL_PATH = Path(
    "LSTM_Original_Keras_INT8.keras"
)

model.quantize("int8")

model.save(
    QUANT_MODEL_PATH
)

print("INT8 quantization completed.")
print(
    f"Quantized model size: "
    f"{QUANT_MODEL_PATH.stat().st_size / 1024:.3f} KB"
)

ValueError: Layer 'dense_14' is already quantized with dtype_policy='int8_from_float32'. Received: mode=int8

In [4]:
import keras
from pathlib import Path

PROJECT_ROOT = Path(r"C:\tinyml-fdia-windows")
MODEL_PATH = PROJECT_ROOT / "models" / "keras" / "LSTM_HPO.keras"

model_int8 = keras.models.load_model(
    MODEL_PATH,
    compile=False
)

model_int8.quantize("int8")

QUANT_PATH = Path("LSTM_Original_Keras_INT8.keras")
model_int8.save(QUANT_PATH)

print(f"Saved: {QUANT_PATH}")
print(f"Size: {QUANT_PATH.stat().st_size / 1024:.3f} KB")

Saved: LSTM_Original_Keras_INT8.keras
Size: 623.920 KB


In [5]:
for layer in model_int8.layers:
    print(
        layer.name,
        layer.__class__.__name__,
        layer.dtype_policy
    )

lstm_6 LSTM <DTypePolicy "float32">
lstm_7 LSTM <DTypePolicy "float32">
lstm_8 LSTM <DTypePolicy "float32">
dense_14 Dense <QuantizedDTypePolicy "int8_from_float32">
dense_15 Dense <QuantizedDTypePolicy "int8_from_float32">
dense_16 Dense <QuantizedDTypePolicy "int8_from_float32">
dense_17 Dense <QuantizedDTypePolicy "int8_from_float32">
dense_18 Dense <QuantizedDTypePolicy "int8_from_float32">
dense_19 Dense <QuantizedDTypePolicy "int8_from_float32">
dense_20 Dense <QuantizedDTypePolicy "int8_from_float32">


In [6]:
import timeit
import numpy as np
import tensorflow as tf

# Same 10% timing subset
rng = np.random.default_rng(42)

timing_indices = rng.choice(
    len(X_test_lstm),
    size=int(len(X_test_lstm) * 0.10),
    replace=False
)

X_timing = X_test_lstm[timing_indices]


@tf.function
def infer(sample):
    return model(sample, training=False)


# Warm-up
for sample in X_timing[:10]:
    sample = sample[np.newaxis, ...].astype(np.float32)
    _ = infer(sample)


# model.predict()
predict_times = []

for sample in X_timing:
    sample = sample[np.newaxis, ...].astype(np.float32)

    start = timeit.default_timer()

    output = model.predict(
        sample,
        verbose=0
    )

    end = timeit.default_timer()

    predict_times.append(
        (end - start) * 1000
    )


# tf.function
function_times = []

for sample in X_timing:
    sample = sample[np.newaxis, ...].astype(np.float32)

    start = timeit.default_timer()

    output = infer(sample)
    _ = output.numpy()

    end = timeit.default_timer()

    function_times.append(
        (end - start) * 1000
    )


print("model.predict()")
print(f"Mean: {np.mean(predict_times):.6f} ms")
print(f"Std:  {np.std(predict_times):.6f} ms")

print("\ntf.function")
print(f"Mean: {np.mean(function_times):.6f} ms")
print(f"Std:  {np.std(function_times):.6f} ms")

print(
    f"\nSpeedup: "
    f"{np.mean(predict_times) / np.mean(function_times):.2f}x"
)

model.predict()
Mean: 108.168468 ms
Std:  30.052111 ms

tf.function
Mean: 14.809855 ms
Std:  1.588929 ms

Speedup: 7.30x


In [1]:
from pathlib import Path
import tensorflow as tf
from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2

PROJECT_ROOT = Path(r"C:\tinyml-fdia-windows")
KERAS_DIR = PROJECT_ROOT / "models" / "keras"
TFLITE_DIR = PROJECT_ROOT / "models" / "tflite_new"

TFLITE_DIR.mkdir(parents=True, exist_ok=True)

models = {
    "LSTM_Original": "LSTM_HPO.keras",
    "LSTM_Node": "LSTM_NodePruned.keras",
    "LSTM_Weight": "LSTM_WeightPruned.keras",
    "LSTM_WeightNode": "LSTM_WeightNodePruned.keras",
    "LSTM_NodeWeight": "LSTM_NodeWeightPruned.keras",
}

for name, filename in models.items():
    print(f"\nConverting: {name}")

    model = tf.keras.models.load_model(KERAS_DIR / filename, compile=False)

    @tf.function(input_signature=[tf.TensorSpec(shape=[1, 83, 6], dtype=tf.float32)])
    def inference_function(x):
        return model(x, training=False)

    concrete_function = inference_function.get_concrete_function()
    frozen_function = convert_variables_to_constants_v2(concrete_function)

    # Baseline FP32
    converter = tf.lite.TFLiteConverter.from_concrete_functions([frozen_function])
    baseline_model = converter.convert()

    baseline_path = TFLITE_DIR / f"{name}_baseline.tflite"
    baseline_path.write_bytes(baseline_model)

    # Dynamic range quantization
    converter = tf.lite.TFLiteConverter.from_concrete_functions([frozen_function])
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    quant_model = converter.convert()

    quant_path = TFLITE_DIR / f"{name}_quant.tflite"
    quant_path.write_bytes(quant_model)

    print(f"Baseline: {baseline_path.stat().st_size / 1024:.3f} KB")
    print(f"Dynamic:  {quant_path.stat().st_size / 1024:.3f} KB")


Converting: LSTM_Original


Baseline: 681.566 KB
Dynamic:  246.031 KB

Converting: LSTM_Node


Baseline: 596.055 KB
Dynamic:  233.289 KB

Converting: LSTM_Weight


Baseline: 680.570 KB
Dynamic:  245.023 KB

Converting: LSTM_WeightNode


Baseline: 663.871 KB
Dynamic:  256.609 KB

Converting: LSTM_NodeWeight


Baseline: 596.301 KB
Dynamic:  233.539 KB


In [2]:
from pathlib import Path
import numpy as np
import tensorflow as tf

TFLITE_DIR = Path(r"C:\tinyml-fdia-windows\models\tflite_new")

for model_path in sorted(TFLITE_DIR.glob("*.tflite")):
    interpreter = tf.lite.Interpreter(model_path=str(model_path))
    interpreter.allocate_tensors()

    tensor_details = interpreter.get_tensor_details()
    input_indices = {x["index"] for x in interpreter.get_input_details()}
    output_indices = {x["index"] for x in interpreter.get_output_details()}

    total_params = 0
    float32_params = 0
    int8_params = 0
    parameter_bytes = 0

    for tensor in tensor_details:
        index = tensor["index"]

        if index in input_indices or index in output_indices:
            continue

        shape = tensor["shape"]

        if len(shape) == 0:
            continue

        try:
            values = interpreter.get_tensor(index)
        except ValueError:
            continue

        if values.size == 0:
            continue

        count = values.size
        dtype = values.dtype

        total_params += count
        parameter_bytes += values.nbytes

        if dtype == np.float32:
            float32_params += count

        elif dtype == np.int8:
            int8_params += count

    file_size_kb = model_path.stat().st_size / 1024
    parameter_size_kb = parameter_bytes / 1024

    print(f"\n{model_path.name}")
    print(f"Estimated parameters: {total_params:,}")
    print(f"Float32 parameters:   {float32_params:,}")
    print(f"Int8 parameters:      {int8_params:,}")
    print(f"Parameter storage:    {parameter_size_kb:.3f} KB")
    print(f"TFLite file size:     {file_size_kb:.3f} KB")

c:\tinyml-fdia-windows\venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



LSTM_Node_baseline.tflite
Estimated parameters: 31,057
Float32 parameters:   31,045
Int8 parameters:      0
Parameter storage:    121.316 KB
TFLite file size:     596.055 KB

LSTM_Node_quant.tflite
Estimated parameters: 31,057
Float32 parameters:   12,421
Int8 parameters:      18,624
Parameter storage:    66.754 KB
TFLite file size:     233.289 KB

LSTM_NodeWeight_baseline.tflite
Estimated parameters: 31,057
Float32 parameters:   31,045
Int8 parameters:      0
Parameter storage:    121.316 KB
TFLite file size:     596.301 KB

LSTM_NodeWeight_quant.tflite
Estimated parameters: 31,057
Float32 parameters:   12,421
Int8 parameters:      18,624
Parameter storage:    66.754 KB
TFLite file size:     233.539 KB

LSTM_Original_baseline.tflite
Estimated parameters: 27,291
Float32 parameters:   27,279
Int8 parameters:      0
Parameter storage:    106.605 KB
TFLite file size:     681.566 KB

LSTM_Original_quant.tflite
Estimated parameters: 27,291
Float32 parameters:   8,591
Int8 parameters:      

In [3]:
from pathlib import Path
import numpy as np
import tensorflow as tf

TFLITE_DIR = Path(r"C:\tinyml-fdia-windows\models\tflite_new")

def inspect_tflite_weights(model_path):
    model_bytes = model_path.read_bytes()
    model = tf.lite.experimental.Analyzer

    interpreter = tf.lite.Interpreter(model_path=str(model_path))
    interpreter.allocate_tensors()

    details = interpreter.get_tensor_details()

    float32_count = 0
    int8_count = 0
    int32_count = 0
    other_count = 0

    float32_bytes = 0
    int8_bytes = 0
    int32_bytes = 0
    other_bytes = 0

    print(f"\n{model_path.name}")
    print("-" * 70)

    for tensor in details:
        try:
            values = interpreter.get_tensor(tensor["index"])
        except:
            continue

        if values.size == 0:
            continue

        name = tensor["name"]
        dtype = values.dtype
        count = values.size
        size_bytes = values.nbytes

        if dtype == np.float32:
            float32_count += count
            float32_bytes += size_bytes
        elif dtype == np.int8:
            int8_count += count
            int8_bytes += size_bytes
        elif dtype == np.int32:
            int32_count += count
            int32_bytes += size_bytes
        else:
            other_count += count
            other_bytes += size_bytes

        if dtype in [np.float32, np.int8, np.int32]:
            print(f"{str(dtype):10s} | {count:8,d} | {size_bytes/1024:8.3f} KB | {name}")

    print("\nSummary")
    print(f"Float32: {float32_count:,} values -> {float32_bytes / 1024:.3f} KB")
    print(f"Int8:    {int8_count:,} values -> {int8_bytes / 1024:.3f} KB")
    print(f"Int32:   {int32_count:,} values -> {int32_bytes / 1024:.3f} KB")
    print(f"Other:   {other_count:,} values -> {other_bytes / 1024:.3f} KB")


for model_path in sorted(TFLITE_DIR.glob("*quant.tflite")):
    inspect_tflite_weights(model_path)


LSTM_Node_quant.tflite
----------------------------------------------------------------------
float32    |      498 |    1.945 KB | x
int32      |        3 |    0.012 KB | sequential_2_1/lstm_3_1/transpose
int32      |        1 |    0.004 KB | sequential_2_1/lstm_3_1/while/cond/_0/sequential_2_1/lstm_3_1/while/Less/y
int32      |        1 |    0.004 KB | arith.constant
float32    |      204 |    0.797 KB | sequential_2_1/lstm_3_1/lstm_cell_1/BiasAdd_1/ReadVariableOp/resource
float32    |       51 |    0.199 KB | arith.constant1
float32    |      284 |    1.109 KB | sequential_2_1/lstm_4_1/lstm_cell_1/BiasAdd_1/ReadVariableOp/resource
float32    |       71 |    0.277 KB | arith.constant2
float32    |      396 |    1.547 KB | sequential_2_1/lstm_5_1/lstm_cell_1/BiasAdd_1/ReadVariableOp/resource
float32    |       99 |    0.387 KB | arith.constant3
float32    |    4,233 |   16.535 KB | sequential_2_1/lstm_3_1/TensorArrayV2_1
float32    |    5,893 |   23.020 KB | sequential_2_1/lstm_4_1/T

In [4]:
from pathlib import Path
from collections import defaultdict
import numpy as np
from tensorflow.lite.python import schema_py_generated as schema_fb

MODEL_PATH = Path(r"C:\tinyml-fdia-windows\models\tflite_new\LSTM_Original_quant.tflite")

# TFLite tensor type -> readable name and bytes per element
TYPE_INFO = {
    schema_fb.TensorType.FLOAT32: ("FLOAT32", 4),
    schema_fb.TensorType.FLOAT16: ("FLOAT16", 2),
    schema_fb.TensorType.INT8: ("INT8", 1),
    schema_fb.TensorType.UINT8: ("UINT8", 1),
    schema_fb.TensorType.INT16: ("INT16", 2),
    schema_fb.TensorType.INT32: ("INT32", 4),
    schema_fb.TensorType.INT64: ("INT64", 8),
    schema_fb.TensorType.BOOL: ("BOOL", 1),
}

model_bytes = MODEL_PATH.read_bytes()
model = schema_fb.Model.GetRootAsModel(model_bytes, 0)

summary = defaultdict(lambda: {"elements": 0, "bytes": 0, "buffers": set()})

print(f"Model: {MODEL_PATH.name}")
print(f"Subgraphs: {model.SubgraphsLength()}")
print("=" * 100)

for sg_idx in range(model.SubgraphsLength()):
    subgraph = model.Subgraphs(sg_idx)

    sg_name = subgraph.Name()
    sg_name = sg_name.decode("utf-8") if sg_name else f"subgraph_{sg_idx}"

    print(f"\nSUBGRAPH {sg_idx}: {sg_name}")
    print("-" * 100)

    for tensor_idx in range(subgraph.TensorsLength()):
        tensor = subgraph.Tensors(tensor_idx)

        buffer_idx = tensor.Buffer()
        buffer = model.Buffers(buffer_idx)

        # No stored data = activation/intermediate tensor, not a stored constant
        if buffer is None or buffer.DataLength() == 0:
            continue

        name = tensor.Name()
        name = name.decode("utf-8") if name else f"tensor_{tensor_idx}"

        shape = [tensor.Shape(i) for i in range(tensor.ShapeLength())]
        elements = int(np.prod(shape)) if shape else 1

        dtype, bytes_per_element = TYPE_INFO.get(
            tensor.Type(),
            (f"TYPE_{tensor.Type()}", None)
        )

        stored_bytes = buffer.DataLength()

        summary[dtype]["elements"] += elements
        summary[dtype]["bytes"] += stored_bytes
        summary[dtype]["buffers"].add(buffer_idx)

        print(
            f"{dtype:8s} | "
            f"shape={str(shape):20s} | "
            f"elements={elements:8,d} | "
            f"buffer={stored_bytes / 1024:8.3f} KB | "
            f"{name}"
        )

print("\n" + "=" * 100)
print("SUMMARY OF STORED CONSTANT TENSORS")
print("=" * 100)

for dtype, values in summary.items():
    print(
        f"{dtype:8s} | "
        f"elements={values['elements']:10,d} | "
        f"stored={values['bytes'] / 1024:9.3f} KB | "
        f"buffers={len(values['buffers'])}"
    )

print(f"\nComplete TFLite file: {MODEL_PATH.stat().st_size / 1024:.3f} KB")

Model: LSTM_Original_quant.tflite
Subgraphs: 7

SUBGRAPH 0: main
----------------------------------------------------------------------------------------------------
INT32    | shape=[3]                  | elements=       3 | buffer=   0.012 KB | sequential_2_1/lstm_6_1/transpose
INT32    | shape=[]                   | elements=       1 | buffer=   0.004 KB | sequential_2_1/lstm_6_1/while/cond/_0/sequential_2_1/lstm_6_1/while/Less/y
INT32    | shape=[]                   | elements=       1 | buffer=   0.004 KB | arith.constant
FLOAT32  | shape=[300]                | elements=     300 | buffer=   1.172 KB | sequential_2_1/lstm_6_1/lstm_cell_1/BiasAdd_1/ReadVariableOp/resource
FLOAT32  | shape=[1, 75]              | elements=      75 | buffer=   0.293 KB | arith.constant1
FLOAT32  | shape=[300]                | elements=     300 | buffer=   1.172 KB | sequential_2_1/lstm_7_1/lstm_cell_1/BiasAdd_1/ReadVariableOp/resource
FLOAT32  | shape=[400]                | elements=     400 | buffer= 